# Medical Appointment No-Show Analysis

## Business Objective

The objective of this project is to analyze medical appointment records to identify patterns in patient attendance and missed appointments (No-shows). The analysis explores patient demographics, appointment scheduling, medical conditions, and other factors that may influence whether a patient attends or misses a scheduled appointment. The insights generated can help healthcare providers better understand appointment behavior and support data-driven decision-making for improving appointment management.


## Project Objective

The objective of this project is to analyze medical appointment records to understand patient attendance behavior. The analysis focuses on identifying patterns in appointment attendance, patient demographics, scheduling trends, and healthcare-related factors that may influence whether patients attend or miss their appointments.


In [5]:
import pandas as pd

### Data Inspection 

In [6]:
df = pd.read_csv("KaggleV2-May-2016.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include="object")

In [8]:
#Check for Duplicate Rows
df.duplicated().sum()

np.int64(0)

In [25]:
#Check for Missing Value
df.isnull().sum()

PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0
dtype: int64

In [24]:
df.dtypes

PatientId                     float64
AppointmentID                   int64
Gender                         object
ScheduledDay      datetime64[ns, UTC]
AppointmentDay    datetime64[ns, UTC]
Age                             int64
Neighbourhood                  object
Scholarship                     int64
Hipertension                    int64
Diabetes                        int64
Alcoholism                      int64
Handcap                         int64
SMS_received                    int64
No-show                        object
dtype: object

## Data Cleaning

Data cleaning is an essential step before analysis. In this section, the dataset is examined for data quality issues such as duplicate records, missing values, inconsistent entries, and incorrect data types. Appropriate cleaning techniques are then applied to improve the accuracy and consistency of the analysis.

In [11]:
#Convert Date Columns "ScheduledDay" to real panda datetime

df["ScheduledDay"] = pd.to_datetime(df["ScheduledDay"])

In [12]:
#Convert Date Columns "AppointmentDay" to real panda datetime

df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"])

In [ ]:
#Verify Data Types Again to confirm if any age is less than 0
df[df["Age"] < 0]

In [18]:
# We found age that is -0, Since age cannot be negative, this record is invalid and should be removed. 

#Clean the invalid record ansd applying back to dataframe
df = df[df["Age"] >= 0]

In [ ]:
#Verify the result
df[df["Age"] < 0]

In [20]:
df["Gender"].unique()

array(['F', 'M'], dtype=object)

In [21]:
df["No-show"].unique()

array(['No', 'Yes'], dtype=object)

In [22]:
df["Neighbourhood"].nunique()

81

In [23]:
df["Neighbourhood"].sort_values().unique()

array(['AEROPORTO', 'ANDORINHAS', 'ANTÔNIO HONÓRIO',
       'ARIOVALDO FAVALESSA', 'BARRO VERMELHO', 'BELA VISTA',
       'BENTO FERREIRA', 'BOA VISTA', 'BONFIM', 'CARATOÍRA', 'CENTRO',
       'COMDUSA', 'CONQUISTA', 'CONSOLAÇÃO', 'CRUZAMENTO', 'DA PENHA',
       'DE LOURDES', 'DO CABRAL', 'DO MOSCOSO', 'DO QUADRO',
       'ENSEADA DO SUÁ', 'ESTRELINHA', 'FONTE GRANDE', 'FORTE SÃO JOÃO',
       'FRADINHOS', 'GOIABEIRAS', 'GRANDE VITÓRIA', 'GURIGICA', 'HORTO',
       'ILHA DAS CAIEIRAS', 'ILHA DE SANTA MARIA', 'ILHA DO BOI',
       'ILHA DO FRADE', 'ILHA DO PRÍNCIPE', 'ILHAS OCEÂNICAS DE TRINDADE',
       'INHANGUETÁ', 'ITARARÉ', 'JABOUR', 'JARDIM CAMBURI',
       'JARDIM DA PENHA', 'JESUS DE NAZARETH', 'JOANA D´ARC',
       'JUCUTUQUARA', 'MARIA ORTIZ', 'MARUÍPE', 'MATA DA PRAIA',
       'MONTE BELO', 'MORADA DE CAMBURI', 'MÁRIO CYPRESTE', 'NAZARETH',
       'NOVA PALESTINA', 'PARQUE INDUSTRIAL', 'PARQUE MOSCOSO', 'PIEDADE',
       'PONTAL DE CAMBURI', 'PRAIA DO CANTO', 'PRAIA DO SUÁ',

## Exploratory Data Analysis (EDA)

#### Question 1
How many medical appointments are recorded in the dataset?

In [27]:
df["AppointmentID"].count()

np.int64(110526)


The dataset contains **110,526** medical appointment records after data cleaning. This represents the total number of appointments available for analysis and serves as the basis for all subsequent exploratory analyses.

#### Question 2

How many unique patients are represented in the dataset?

In [29]:
df["PatientId"].nunique()

62298

The dataset contains **62,298** unique patients who collectively made **110,526** medical appointments. This indicates that many patients scheduled more than one appointment during the period covered by the dataset.

#### Question 3

What is the distribution of appointment attendance (No-show vs Attended)?

In [32]:
df["No-show"].value_counts()

No-show
No     88207
Yes    22319
Name: count, dtype: int64

Out of **110,526** medical appointments, **88,207** were attended, while **22,319** resulted in a No-show. This indicates that the majority of patients attended their scheduled appointments, although a substantial number of appointments were still missed.

#### Question 4

What percentage of appointments resulted in a No-show versus attendance?

In [33]:
df["No-show"].value_counts(normalize="index") * 100

No-show
No     79.806561
Yes    20.193439
Name: proportion, dtype: float64

Approximately **79.81%** of medical appointments were attended, while **20.19%** resulted in a No-show. This indicates that although most patients kept their appointments, about **1 in every 5** scheduled appointments was missed, highlighting an opportunity to improve appointment attendance.

#### Question 5

What is the gender distribution of patients?

In [34]:
df["Gender"].value_counts()

Gender
F    71839
M    38687
Name: count, dtype: int64

The dataset contains **71,839** female patients and **38,687** male patients. This indicates that female patients accounted for a larger proportion of medical appointments than male patients during the period covered by the dataset.

#### Question 6

What is the average age of patients?

In [37]:
df["Age"].mean()

np.float64(37.089218826339504)

The average age of patients is approximately **37.09 years**, indicating that the dataset is primarily composed of adults in their mid-thirties. This provides a general overview of the age profile of patients who scheduled medical appointments.

#### Question 7

Which age group has the highest number of appointments?

In [49]:
# We first need to create a group by categorizing  column Age as (Age_Group) using pd.cut
# Assign the category valuen inside (Age_Group)

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0, 12, 19, 35, 60, 115], #cut from 0 to 12 for child, 12 to 19 for teenager, and the likes
    labels=["Child", "Teenager", "Young Adult", "Adult", "Senior"], #lael for each grouping category
    include_lowest=True #include eve the lowest values
)

df["Age_Group"].value_counts() #Age_Group tells us the answer

Age_Group
Adult          37761
Young Adult    22592
Child          21036
Senior         19762
Teenager        9375
Name: count, dtype: int64

The **Adult** age group recorded the highest number of medical appointments with **37,761** appointments, followed by **Young Adults** with **22,592** and **Children** with **21,036**. **Seniors** accounted for **19,762** appointments, while **Teenagers** had the fewest at **9,375**. This indicates that adults represented the largest proportion of patients seeking medical appointments during the study period.

#### Question 8
Does receiving an SMS reminder appear to influence appointment attendance?

In [57]:
pd.crosstab(df["SMS_received"], df["No-show"])

#or for percentage %

pd.crosstab(df["SMS_received"], df["No-show"], normalize="index") * 100

No-show,No,Yes
SMS_received,,
0,83.296466,16.703534
1,72.425455,27.574545


Patients who **did not receive** an SMS reminder had an attendance rate of **83.30%**, while **16.70%** missed their appointments. In contrast, patients who **received** an SMS reminder had an attendance rate of **72.43%**, while **27.57%** did not attend. Based on this dataset, patients who received an SMS reminder had a higher proportion of missed appointments than those who did not receive one. However, this analysis shows an association only and does not establish that receiving an SMS reminder caused the higher no-show rate.

#### Question 9
Do patients enrolled in the scholarship program have different attendance patterns compared to those not enrolled?

In [58]:
pd.crosstab(df["Scholarship"], df["No-show"], normalize="index") * 100

No-show,No,Yes
Scholarship,,
0,80.192645,19.807355
1,76.263696,23.736304


Patients **not enrolled** in the scholarship program had an attendance rate of **80.19%**, while **19.81%** missed their appointments. Patients **enrolled** in the scholarship program had an attendance rate of **76.26%**, with **23.74%** missing their appointments. Based on this dataset, patients enrolled in the scholarship program had a higher proportion of missed appointments than those not enrolled. However, this analysis indicates an association only and does not establish that scholarship enrollment caused the difference in attendance.

#### Question 10
Which neighbourhood has the highest number of missed appointments (No-shows)?

In [62]:
df[df["No-show"] == "Yes"]["Neighbourhood"].value_counts()

Neighbourhood
JARDIM CAMBURI                 1465
MARIA ORTIZ                    1219
ITARARÉ                         923
RESISTÊNCIA                     906
CENTRO                          703
                               ... 
PONTAL DE CAMBURI                12
ILHA DO BOI                       3
ILHAS OCEÂNICAS DE TRINDADE       2
ILHA DO FRADE                     2
AEROPORTO                         1
Name: count, Length: 80, dtype: int64

The neighbourhood with the highest number of missed appointments was **JARDIM CAMBURI**, recording **1,465** no-shows. This was followed by **MARIA ORTIZ** (**1,219**), **ITARARÉ** (**923**), **RESISTÊNCIA** (**906**), and **CENTRO** (**703**). These neighbourhoods recorded the largest numbers of missed appointments in the dataset.

## Key Insights

- The dataset contains **110,526** medical appointments from **62,298** unique patients.
- **79.81%** of appointments were attended, while **20.19%** resulted in a No-show.
- Female patients accounted for the majority of appointments, with **71,839** appointments compared to **38,687** for male patients.
- The average patient age was approximately **37.09 years**.
- The **Adult** age group recorded the highest number of appointments (**37,761**), followed by **Young Adults** (**22,592**).
- Patients who received an SMS reminder had a higher proportion of missed appointments (**27.57%**) than those who did not receive an SMS reminder (**16.70%**). This represents an observed association and does not imply causation.
- Patients enrolled in the scholarship program recorded a slightly higher proportion of missed appointments (**23.74%**) than patients not enrolled (**19.81%**).
- **JARDIM CAMBURI** recorded the highest number of missed appointments (**1,465**), followed by **MARIA ORTIZ** (**1,219**) and **ITARARÉ** (**923**).

## Business Recommendations

- Monitor neighbourhoods with the highest numbers of missed appointments, such as **JARDIM CAMBURI**, to better understand local attendance patterns and allocate follow-up efforts where they are most needed.
- Review the effectiveness of the current SMS reminder process since patients who received reminders recorded a higher proportion of missed appointments. Additional investigation may help determine whether reminder timing or other factors influence attendance.
- Pay closer attention to patient groups with relatively higher no-show rates, such as scholarship beneficiaries, to better understand potential barriers to appointment attendance.
- Continue tracking appointment attendance across different age groups and patient characteristics to support data-driven planning and resource allocation.

## Conclusion

This analysis explored medical appointment attendance using patient demographics, appointment characteristics, and selected health-related variables. Most appointments were successfully attended, although approximately one-fifth resulted in a No-show. Differences in attendance patterns were observed across SMS reminder status, scholarship enrollment, age groups, and neighbourhoods. These findings provide useful descriptive insights that can support healthcare providers in monitoring appointment attendance and identifying areas for further investigation.````